# Notebook 08 — Evaluation & Ablation Study
5-configuration ablation on the full test set. This is the paper results section.

In [ ]:
import os, sys, json, time, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              precision_score, recall_score)

COLAB_BASE = "/content/drive/MyDrive/PAS"
if os.path.exists(COLAB_BASE):
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    BASE = COLAB_BASE
else:
    BASE = str(Path(".").resolve().parent)

for p in [os.path.join(BASE, "AGENTS"), str(Path(".").resolve().parent / "AGENTS")]:
    if p not in sys.path: sys.path.insert(0, p)

CACHE_DIR   = os.path.join(BASE, "EXPERIMENT_CACHE")
MODELS_DIR  = os.path.join(BASE, "MODELS")
RESULTS_DIR = os.path.join(BASE, "RESULTS")
ABLATION_DIR = os.path.join(RESULTS_DIR, "ablation"); os.makedirs(ABLATION_DIR, exist_ok=True)
LATENCY_DIR  = os.path.join(RESULTS_DIR, "latency"); os.makedirs(LATENCY_DIR, exist_ok=True)
print("Setup OK")

In [ ]:
df = pd.read_parquet(os.path.join(CACHE_DIR, "features.parquet"))
test_df = df[df["split"] == "test"].copy().reset_index(drop=True)
y_true = test_df["label_int"].values
print(f"Test set: {len(test_df):,} samples | Malicious: {y_true.mean():.2%}")

def compute_all_metrics(y_true, y_pred, y_prob=None, name=""):
    fnr = ((y_true == 1) & (y_pred == 0)).sum() / max(1, (y_true == 1).sum())
    fpr = ((y_true == 0) & (y_pred == 1)).sum() / max(1, (y_true == 0).sum())
    m = {
        "Config": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-Macro": f1_score(y_true, y_pred, average="macro"),
        "F1-Weighted": f1_score(y_true, y_pred, average="weighted"),
        "ROC-AUC": roc_auc_score(y_true, y_prob) if y_prob is not None else float("nan"),
        "FNR": fnr,
        "FPR": fpr,
    }
    return m

results = []
print("Metrics function loaded.")

In [ ]:
# Config 1: Prompt Agent only (threshold on malicious_probability)
prob_col = "malicious_probability"
y_prob_1 = test_df[prob_col].values
y_pred_1 = (y_prob_1 >= 0.5).astype(int)
results.append(compute_all_metrics(y_true, y_pred_1, y_prob_1, "C1: Prompt Agent Only"))
print("C1 done:", results[-1]["F1-Macro"])

# Config 2: Vision Agent only (threshold on vision_score)
if "vision_score" in test_df.columns:
    y_prob_2 = test_df["vision_score"].values
    y_pred_2 = (y_prob_2 >= 0.35).astype(int)
    results.append(compute_all_metrics(y_true, y_pred_2, y_prob_2, "C2: Vision Agent Only"))
print("C2 done")

# Config 3: Prompt + Vision (max fusion without risk model)
y_prob_3 = np.maximum(test_df["malicious_probability"].values,
                       test_df.get("vision_score", pd.Series(0, index=test_df.index)).values)
y_pred_3 = (y_prob_3 >= 0.5).astype(int)
results.append(compute_all_metrics(y_true, y_pred_3, y_prob_3, "C3: Prompt + Vision (max)"))
print("C3 done")

# Config 4: Full risk model (Logistic Regression)
if "risk_score" in test_df.columns:
    y_prob_4 = test_df["risk_score"].values
    y_pred_4 = (y_prob_4 >= 0.5).astype(int)
    results.append(compute_all_metrics(y_true, y_pred_4, y_prob_4, "C4: Risk Model Fusion"))
print("C4 done")

# Config 5: Full framework (governance decisions mapped to binary)
if "decision" in test_df.columns:
    y_pred_5 = (test_df["decision"].isin(["BLOCK", "SANITIZE"])).astype(int)
    y_prob_5 = test_df.get("risk_score", test_df["malicious_probability"]).values
    results.append(compute_all_metrics(y_true, y_pred_5, y_prob_5, "C5: Full Framework"))
print("C5 done")

abl_df = pd.DataFrame(results)
abl_df.to_csv(os.path.join(ABLATION_DIR, "ablation_results.csv"), index=False)
display(abl_df.round(4))

In [ ]:
# Main ablation table (paper Table 2)
fig, ax = plt.subplots(figsize=(14, 4))
ax.axis("off")
table = ax.table(
    cellText=abl_df.round(4).values,
    colLabels=abl_df.columns,
    cellLoc="center", loc="center"
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.8)
plt.title("Ablation Study \u2014 5 Configurations", fontsize=13, fontweight="bold", pad=20)
plt.tight_layout()
plt.savefig(os.path.join(ABLATION_DIR, "08_ablation_table.png"), bbox_inches="tight", dpi=200)
plt.show()

# FNR comparison (most critical metric)
fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#e74c3c" if v > 0.1 else "#2ecc71" for v in abl_df["FNR"]]
bars = ax.bar(abl_df["Config"], abl_df["FNR"] * 100, color=colors, edgecolor="white")
ax.axhline(10, color="black", linestyle="--", label="10% safety threshold")
for bar, v in zip(bars, abl_df["FNR"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f"{v:.1%}", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax.set_ylabel("False Negative Rate (%)")
ax.set_title("FNR per Configuration (Lower = Safer)", fontweight="bold")
ax.legend()
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(ABLATION_DIR, "08_fnr_comparison.png"), bbox_inches="tight", dpi=150)
plt.show()